In [1]:
   import duckdb

   DATA = "../../data/raw/olist"

   views = {
       "orders": "olist_orders_dataset.csv",
       "customers": "olist_customers_dataset.csv",
       "items": "olist_order_items_dataset.csv",
       "payments": "olist_order_payments_dataset.csv",
       "products": "olist_products_dataset.csv",
       "category_translation": "product_category_name_translation.csv",
   }

   for name, file in views.items():
       duckdb.sql(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_csv('{DATA}/{file}')")

In [2]:
   duckdb.sql("SELECT COUNT(*) AS rows, COUNT(DISTINCT order_id) AS different_ids FROM orders")
   
   

┌───────┬───────────────┐
│ rows  │ different_ids │
│ int64 │     int64     │
├───────┼───────────────┤
│ 99441 │         99441 │
└───────┴───────────────┘

In [6]:
  duckdb.sql("""
       SELECT o.order_id, o.order_status, c.customer_state
       FROM orders AS o
       INNER JOIN customers AS c ON o.customer_id = c.customer_id
       
       LIMIT 5
   """)

┌──────────────────────────────────┬──────────────┬────────────────┐
│             order_id             │ order_status │ customer_state │
│             varchar              │   varchar    │    varchar     │
├──────────────────────────────────┼──────────────┼────────────────┤
│ e481f51cbdc54678b7cc49136f2d6af7 │ delivered    │ SP             │
│ 53cdb2fc8bc7dce0b6741e2150273451 │ delivered    │ BA             │
│ 47770eb9100c2d0c44946d9cf07ec65d │ delivered    │ GO             │
│ ad21c59c0840e6cb83a9ceb5573f8159 │ delivered    │ SP             │
│ a4591c265e18cb1dcee52889e2d8acc3 │ delivered    │ PR             │
└──────────────────────────────────┴──────────────┴────────────────┘

In [5]:
   duckdb.sql("""
       SELECT COUNT(*)
       FROM products AS p
       INNER JOIN category_translation AS t ON p.product_category_name = t.product_category_name
   """)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        32328 │
└──────────────┘

In [7]:
   duckdb.sql("""
       SELECT COUNT(*)
       FROM products AS p
       LEFT JOIN category_translation AS t ON p.product_category_name = t.product_category_name
   """)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        32951 │
└──────────────┘

In [8]:
   duckdb.sql("""
       SELECT p.product_id, p.product_category_name, t.product_category_name_english
       FROM products AS p
       LEFT JOIN category_translation AS t ON p.product_category_name = t.product_category_name
       WHERE t.product_category_name_english IS NULL
       LIMIT 10
   """)

┌──────────────────────────────────┬───────────────────────┬───────────────────────────────┐
│            product_id            │ product_category_name │ product_category_name_english │
│             varchar              │        varchar        │            varchar            │
├──────────────────────────────────┼───────────────────────┼───────────────────────────────┤
│ a41e356c76fab66334f36de622ecbd3a │ NULL                  │ NULL                          │
│ d8dee61c2034d6d075997acef1870e9b │ NULL                  │ NULL                          │
│ 56139431d72cd51f19eb9f7dae4d1617 │ NULL                  │ NULL                          │
│ 46b48281eb6d663ced748f324108c733 │ NULL                  │ NULL                          │
│ 5fb61f482620cb672f5e586bb132eae9 │ NULL                  │ NULL                          │
│ e10758160da97891c2fdcbc35f0f031d │ NULL                  │ NULL                          │
│ 39e3b9b12cd0bf8ee681bbc1c130feb5 │ NULL                  │ NULL     

In [9]:
   duckdb.sql("""
       SELECT p.product_category_name, COUNT(*) AS products
       FROM products AS p
       LEFT JOIN category_translation AS t ON p.product_category_name = t.product_category_name
       WHERE t.product_category_name_english IS NULL
       GROUP BY p.product_category_name
   """)

┌───────────────────────────────────────────────┬──────────┐
│             product_category_name             │ products │
│                    varchar                    │  int64   │
├───────────────────────────────────────────────┼──────────┤
│ pc_gamer                                      │        3 │
│ NULL                                          │      610 │
│ portateis_cozinha_e_preparadores_de_alimentos │       10 │
└───────────────────────────────────────────────┴──────────┘

In [10]:
   duckdb.sql("""
       SELECT
           t.product_category_name_english AS category,
           COUNT(*)                        AS items_sold,
           ROUND(SUM(i.price), 2)          AS revenue
       FROM items AS i
       INNER JOIN products AS p             ON i.product_id = p.product_id
       LEFT JOIN  category_translation AS t ON p.product_category_name = t.product_category_name
       GROUP BY category
       ORDER BY revenue DESC
       LIMIT 10
   """)

┌───────────────────────┬────────────┬────────────┐
│       category        │ items_sold │  revenue   │
│        varchar        │   int64    │   double   │
├───────────────────────┼────────────┼────────────┤
│ health_beauty         │       9670 │ 1258681.34 │
│ watches_gifts         │       5991 │ 1205005.68 │
│ bed_bath_table        │      11115 │ 1036988.68 │
│ sports_leisure        │       8641 │  988048.97 │
│ computers_accessories │       7827 │  911954.32 │
│ furniture_decor       │       8334 │  729762.49 │
│ cool_stuff            │       3796 │  635290.85 │
│ housewares            │       6964 │  632248.66 │
│ auto                  │       4235 │  592720.11 │
│ garden_tools          │       4347 │  485256.46 │
└───────────────────────┴────────────┴────────────┘
  10 rows                               3 columns

In [11]:
   duckdb.sql("""
       SELECT
           CASE
               WHEN order_status = 'delivered' THEN 'completed'
               WHEN order_status IN ('shipped', 'invoiced', 'processing', 'created', 'approved') THEN 'open'
               WHEN order_status IN ('canceled', 'unavailable') THEN 'lost'
               ELSE 'unknown'
           END AS status_group,
           COUNT(*) AS orders
       FROM orders
       GROUP BY status_group
       ORDER BY orders DESC
   """)

┌──────────────┬────────┐
│ status_group │ orders │
│   varchar    │ int64  │
├──────────────┼────────┤
│ completed    │  96478 │
│ open         │   1729 │
│ lost         │   1234 │
└──────────────┴────────┘

In [12]:
   duckdb.sql("""
       SELECT COUNT(*)
       FROM payments
       WHERE payment_value > (SELECT AVG(payment_value) FROM payments)
   """)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        31012 │
└──────────────┘

In [13]:
   duckdb.sql("""
       SELECT items_in_order, COUNT(*) AS orders
       FROM (
           SELECT order_id, COUNT(*) AS items_in_order
           FROM items
           GROUP BY order_id
       ) AS per_order
       GROUP BY items_in_order
       ORDER BY items_in_order
   """)

┌────────────────┬────────┐
│ items_in_order │ orders │
│     int64      │ int64  │
├────────────────┼────────┤
│              1 │  88863 │
│              2 │   7516 │
│              3 │   1322 │
│              4 │    505 │
│              5 │    204 │
│              6 │    198 │
│              7 │     22 │
│              8 │      8 │
│              9 │      3 │
│             10 │      8 │
│             11 │      4 │
│             12 │      5 │
│             13 │      1 │
│             14 │      2 │
│             15 │      2 │
│             20 │      2 │
│             21 │      1 │
└────────────────┴────────┘
  17 rows       2 columns

In [14]:
  duckdb.sql("""
       CREATE OR REPLACE TABLE category_revenue AS
       SELECT
           t.product_category_name_english AS category,
           ROUND(SUM(i.price), 2)          AS revenue
       FROM items AS i
       INNER JOIN products AS p             ON i.product_id = p.product_id
       LEFT JOIN  category_translation AS t ON p.product_category_name = t.product_category_name
       GROUP BY category
   """)

In [15]:
duckdb.sql("DESCRIBE category_revenue")


┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ category    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ revenue     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘